# Data Cleaning Pipeline

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import numpy as np
from tslearn.clustering import KShape
from tslearn.utils import to_time_series_dataset

from pathlib import Path

/home/gopes/Documents/Clustering-And-Forecasting-TimeSeries-PlayingGround/venv/lib/python3.12/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [3]:
file_2023 = Path("../data/raw/sample_23.csv")
data_2023 = pd.read_csv(file_2023)

In [4]:
dates = [c for c in data_2023.columns if c != "ID"]
dates.sort()

long_rows = []
for _, row in data_2023.iterrows():
    for d in dates:
        long_rows.append({"ID": row["ID"], "date": d, "value": row[d]})

data_2023_long = pd.DataFrame(long_rows)
data_2023_long["date"] = pd.to_datetime(data_2023_long["date"])

display(data_2023_long.head())
print("long table shape:", data_2023_long.shape)


,ID,date,value
0,22.0,2023-01-01,13.482
1,22.0,2023-01-02,9.473
2,22.0,2023-01-03,10.146
3,22.0,2023-01-04,10.978
4,22.0,2023-01-05,14.149


long table shape: (6404655, 3)


In [5]:
zero_flags = (
    data_2023_long
    .groupby("ID")["value"]
    .apply(lambda s: (s == 0).all())
)

zero_ids = zero_flags[zero_flags].index.tolist()

data_2023_long_clean = (
    data_2023_long[~data_2023_long["ID"].isin(zero_ids)]
    .reset_index(drop=True)
)


In [6]:
dates_sorted = sorted(data_2023_long_clean['date'].unique())
wide_profiles = (
    data_2023_long_clean
    .pivot_table(index='ID', columns='date', values='value')
    .reindex(columns=dates_sorted)
)
wide_profiles = wide_profiles.ffill(axis=1).bfill(axis=1)
wide_matrix = wide_profiles.to_numpy()
print(f"Wide profile matrix: {wide_profiles.shape[0]} IDs × {wide_profiles.shape[1]} days")
wide_profiles.head()


Wide profile matrix: 17393 IDs × 365 days


date,2023-01-01,2023-01-02,2023-01-03,2023-01-04,2023-01-05,2023-01-06,2023-01-07,2023-01-08,2023-01-09,2023-01-10,...,2023-12-22,2023-12-23,2023-12-24,2023-12-25,2023-12-26,2023-12-27,2023-12-28,2023-12-29,2023-12-30,2023-12-31
ID,,,,,,,,,,,,,,,,,,,,,
22.0,13.482,9.473,10.146,10.978,14.149,11.536,7.767,10.081,9.189,14.919,...,8.4100,11.847,7.501,12.0760,12.5340,9.3860,9.5890,7.2150,8.623,12.769
42.0,46.427,49.369,40.441,38.126,40.902,28.853,23.482,42.429,43.268,36.268,...,36.6710,46.418,33.754,31.4390,25.3150,25.1880,38.3730,28.3060,32.604,51.493
56.0,9.088,9.300,8.860,13.168,8.341,8.592,14.704,13.383,8.189,8.156,...,11.9470,12.490,14.201,16.8190,10.8730,3.4240,0.1420,4.9070,10.655,4.467
58.0,10.040,7.633,11.596,8.036,10.404,6.576,8.617,15.368,8.773,6.134,...,14.1510,20.087,13.762,14.6370,21.0030,25.3060,5.2450,13.2220,13.909,16.507
64.0,2.969,2.427,2.018,2.742,2.118,2.879,1.961,2.161,2.256,3.755,...,2.5045,2.774,2.018,2.1118,2.1118,2.1118,2.1118,2.1118,2.367,2.729


In [7]:
# Apply moving-average smoothing across each ID's daily profile
ma_window = 7  # 7-day smoothing window
smoothed_profiles = (
    wide_profiles
    .T
    .rolling(window=ma_window, min_periods=1, center=True)
    .mean()
    .T
)

smoothed_matrix = smoothed_profiles.to_numpy()
smoothed_profiles.head()

date,2023-01-01,2023-01-02,2023-01-03,2023-01-04,2023-01-05,2023-01-06,2023-01-07,2023-01-08,2023-01-09,2023-01-10,...,2023-12-22,2023-12-23,2023-12-24,2023-12-25,2023-12-26,2023-12-27,2023-12-28,2023-12-29,2023-12-30,2023-12-31
ID,,,,,,,,,,,,,,,,,,,,,
22.0,11.01975,11.6456,11.627333,11.075857,10.590000,10.549429,11.231286,11.928286,11.271000,12.053286,...,10.689000,11.226714,11.022571,10.191857,10.021143,9.560571,10.313143,10.019333,9.51640,9.54900
42.0,43.59075,43.0530,40.686333,38.228571,37.657429,36.785857,36.189714,34.711286,31.740286,32.409857,...,33.819429,35.216143,33.977000,33.879714,32.684714,30.711286,33.245429,33.546500,35.19280,37.69400
56.0,10.10400,9.7514,9.558167,10.293286,10.906857,10.748143,10.647571,10.064286,10.396571,10.482429,...,11.110143,12.485000,12.384143,9.985143,8.979429,8.717286,7.326714,5.744667,4.71900,5.04275
58.0,9.32625,9.5418,9.047500,8.986000,9.747143,9.910000,9.129714,9.337143,8.991714,9.338143,...,16.598000,16.318714,16.826143,16.313000,16.180286,15.297714,15.689857,15.865333,14.83780,12.22075
64.0,2.53900,2.4548,2.525500,2.444857,2.329429,2.305000,2.553143,2.345143,2.331571,2.222714,...,2.489257,2.396943,2.305200,2.249100,2.193000,2.134857,2.236429,2.257200,2.28628,2.32990


In [8]:
# WEM (Weighted Exponential Moving) smoothing across each ID profile
# Uses existing `wide_profiles` and `ma_window` from previous cells
alpha = 2 / (ma_window + 1)

wem_profiles = wide_profiles.T.ewm(alpha=alpha, adjust=False).mean().T
wem_matrix = wem_profiles.to_numpy()

print(f"WEM profile matrix: {wem_profiles.shape[0]} IDs × {wem_profiles.shape[1]} days (alpha={alpha:.4f})")
wem_profiles.head()

WEM profile matrix: 17393 IDs × 365 days (alpha=0.2500)


date,2023-01-01,2023-01-02,2023-01-03,2023-01-04,2023-01-05,2023-01-06,2023-01-07,2023-01-08,2023-01-09,2023-01-10,...,2023-12-22,2023-12-23,2023-12-24,2023-12-25,2023-12-26,2023-12-27,2023-12-28,2023-12-29,2023-12-30,2023-12-31
ID,,,,,,,,,,,,,,,,,,,,,
22.0,13.482,12.47975,11.896313,11.666734,12.287301,12.099476,11.016357,10.782518,10.384138,11.517854,...,11.029240,11.233680,10.300510,10.744383,11.191787,10.740340,10.452505,9.643129,9.388097,10.233323
42.0,46.427,47.16250,45.482125,43.643094,42.957820,39.431615,35.444211,37.190409,38.709806,38.099355,...,34.497216,37.477412,36.546559,35.269669,32.781002,30.882751,32.755314,31.642985,31.883239,36.785679
56.0,9.088,9.14100,9.070750,10.095062,9.656547,9.390410,10.718808,11.384856,10.585892,9.978419,...,9.377731,10.155798,11.167098,12.580074,12.153305,9.970979,7.513734,6.862051,7.810288,6.974466
58.0,10.040,9.43825,9.977688,9.492266,9.720199,8.934149,8.854862,10.483147,10.055610,9.075207,...,16.125096,17.115572,16.277179,15.867134,17.151101,19.189826,15.703619,15.083214,14.789661,15.218996
64.0,2.969,2.83350,2.629625,2.657719,2.522789,2.611842,2.449131,2.377099,2.346824,2.698868,...,2.606619,2.648464,2.490848,2.396086,2.325015,2.271711,2.231733,2.201750,2.243062,2.364547


In [9]:
output_smoothed = Path("../data/processed/smooth_2023.csv")
smoothed_profiles.to_csv(output_smoothed, index=True)

print(f"Saved: {output_smoothed}")

output_ewm = Path("../data/processed/smooth_ewm_2023.csv")
wem_profiles.to_csv(output_ewm, index=True)

print(f"Saved: {output_ewm}")

Saved: ../data/processed/smooth_2023.csv
Saved: ../data/processed/smooth_ewm_2023.csv
